In [2]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(PROJECT_ROOT)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score
from utils import *

In [4]:
df_ = pd.read_pickle('data/panel/cleaned_data_2.pkl').drop_duplicates()

In [5]:
df = df_.copy()
df.head()

,obid,plz,mietekalt,wohnflaeche,etage,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,...,haustier_erlaubt,heizungsart,kategorie_Wohnung,objektzustand,blid,rent_sqm,is_schlafzimmer_imputed,is_parkplatz_imputed,year,month
81,41534430,22587,918.00,114.700000,1,4.0,1.0,1.0,0,1,...,By arrangement,Central heating,Not specified,Completely renovated,Hamburg,8.003488,False,True,2007,5
94,42410574,20251,374.33,48.000000,1,2.0,1.0,1.0,0,1,...,By arrangement,Not specified,Flat,Well-kept,Hamburg,7.798541,False,True,2007,7
127,38404913,22303,565.60,74.000000,4,3.0,1.0,1.0,0,0,...,No,Central heating,Flat,Well-kept,Hamburg,7.643243,False,True,2007,6
162,41026870,22765,284.00,40.410000,4,2.0,1.0,1.0,1,1,...,No,Central heating,Attic flat,Like new,Hamburg,7.027964,False,True,2007,3
171,36771732,20357,373.00,55.439999,2,2.0,1.0,1.0,0,0,...,By arrangement,Self-contained central heating,Flat,Not specified,Hamburg,6.727994,False,True,2007,7


In [7]:
var_types = {
    "obid": "category",
    "plz": "category",
    "mietekalt": "numerical",
    "wohnflaeche": "numerical",
    "etage": "category",
    "zimmeranzahl": "nominal",
    "schlafzimmer": "nominal",
    "badezimmer": "nominal",
    "aufzug": "binary",
    "balkon": "binary",
    "einbaukueche": "binary",
    "foerderung": "category",
    "gaestewc": "binary",
    "garten": "binary",
    "keller": "binary",
    "parkplatz": "binary",
    "ausstattung": "category",
    "haustier_erlaubt": "category",
    "heizungsart": "category",
    "kategorie_Wohnung": "category",
    "objektzustand": "category",
    "blid": "category",
    "rent_sqm": "numerical",
    "year": "nominal",
    "month": "nominal"
}

In [8]:
nominal_cols = [col for col, t in var_types.items() if t == "nominal"]
df[nominal_cols]

,zimmeranzahl,schlafzimmer,badezimmer,year,month
81,4.0,1.0,1.0,2007,5
94,2.0,1.0,1.0,2007,7
127,3.0,1.0,1.0,2007,6
162,2.0,1.0,1.0,2007,3
171,2.0,1.0,1.0,2007,7
...,...,...,...,...,...
4755927,2.0,1.0,1.0,2023,12
4755930,2.0,1.0,1.0,2023,12
4755931,2.0,1.0,1.0,2023,12
4755936,2.0,1.0,1.0,2023,3


In [9]:
binary_cols = [col for col, t in var_types.items() if t == "binary"]
df[binary_cols]

,aufzug,balkon,einbaukueche,gaestewc,garten,keller,parkplatz
81,0,1,1,0,0,0,0
94,0,1,0,0,0,0,0
127,0,0,1,0,0,1,0
162,1,1,1,0,1,1,0
171,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...
4755927,1,1,1,0,0,0,0
4755930,0,1,0,0,0,0,0
4755931,1,1,1,1,0,0,0
4755936,0,1,0,0,0,0,0


In [21]:
X = df.drop(columns=['obid', 'mietekalt', 'plz', 'rent_sqm'])
y = df['mietekalt']

In [22]:
categorical_cols = [col for col, t in var_types.items() if t == "category" and col in X.columns]
df[categorical_cols]

,etage,foerderung,ausstattung,haustier_erlaubt,heizungsart,kategorie_Wohnung,objektzustand,blid
81,1,0,Sophisticated,By arrangement,Central heating,Not specified,Completely renovated,Hamburg
94,1,0,Not specified,By arrangement,Not specified,Flat,Well-kept,Hamburg
127,4,0,Normal,No,Central heating,Flat,Well-kept,Hamburg
162,4,1,Normal,No,Central heating,Attic flat,Like new,Hamburg
171,2,0,Normal,By arrangement,Self-contained central heating,Flat,Not specified,Hamburg
...,...,...,...,...,...,...,...,...
4755927,5,0,Not specified,By arrangement,Floor heating,Attic flat,Well-kept,The Free State of Saxony
4755930,2,0,Not specified,By arrangement,Central heating,Not specified,Not specified,The Free State of Saxony
4755931,4,0,Normal,By arrangement,Not specified,Flat,Well-kept,The Free State of Saxony
4755936,2,0,Not specified,By arrangement,Central heating,Not specified,Not specified,The Free State of Saxony


In [23]:
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X.head()

,wohnflaeche,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,einbaukueche,gaestewc,garten,keller,...,blid_Hesse,blid_Lower Saxony,blid_Mecklenburg-Western Pommerania,blid_North Rhine-Westphalia,blid_Rhineland-Palatine,blid_Saarland,blid_Saxony-Anhalt,blid_Schleswig Holstein,blid_The Free State of Saxony,blid_The Free State of Thuringia
81,114.700000,4.0,1.0,1.0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
94,48.000000,2.0,1.0,1.0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127,74.000000,3.0,1.0,1.0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
162,40.410000,2.0,1.0,1.0,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
171,55.439999,2.0,1.0,1.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=50, random_state=42)

In [28]:
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MSE: 18356.977869037415
MAE: 85.8898610012085
R² Score: 0.8879833299671543


In [29]:
import joblib

joblib.dump(model, 'model.pkl')

['model.pkl']

In [30]:
df['predicted_mietekalt'] = model.predict(X)

In [ ]:
df['difference'] = df['predicted_mietekalt'] - df['mietekalt']

count    2.067056e+06
mean     8.605530e-01
std      7.628917e+01
min     -1.424000e+03
25%     -1.955600e+01
50%      3.650000e-01
75%      2.402700e+01
max      1.329755e+03
Name: difference, dtype: float64

In [32]:
df['difference'].describe().round(0)

count    2067056.0
mean           1.0
std           76.0
min        -1424.0
25%          -20.0
50%            0.0
75%           24.0
max         1330.0
Name: difference, dtype: float64

In [34]:
df['perc_change'] = df['difference'] / df['mietekalt']

In [35]:
df.head()

,obid,plz,mietekalt,wohnflaeche,etage,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,...,objektzustand,blid,rent_sqm,is_schlafzimmer_imputed,is_parkplatz_imputed,year,month,predicted_mietekalt,difference,perc_change
81,41534430,22587,918.00,114.700000,1,4.0,1.0,1.0,0,1,...,Completely renovated,Hamburg,8.003488,False,True,2007,5,1089.3740,171.3740,0.186682
94,42410574,20251,374.33,48.000000,1,2.0,1.0,1.0,0,1,...,Well-kept,Hamburg,7.798541,False,True,2007,7,363.7638,-10.5662,-0.028227
127,38404913,22303,565.60,74.000000,4,3.0,1.0,1.0,0,0,...,Well-kept,Hamburg,7.643243,False,True,2007,6,583.2398,17.6398,0.031188
162,41026870,22765,284.00,40.410000,4,2.0,1.0,1.0,1,1,...,Like new,Hamburg,7.027964,False,True,2007,3,324.5168,40.5168,0.142665
171,36771732,20357,373.00,55.439999,2,2.0,1.0,1.0,0,0,...,Not specified,Hamburg,6.727994,False,True,2007,7,394.1040,21.1040,0.056579


In [36]:
df.perc_change = df.perc_change.round(2)
df.difference = df.difference.round(2)
df.predicted_mietekalt = df.predicted_mietekalt.round(2)

In [37]:
df.to_csv('predicted_data_3.csv', index=False)

In [39]:
df.to_pickle('predicted_data_3.pkl')